# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainabaon/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zainabaon/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

os.makedirs("work/outputs", exist_ok=True)
print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship


In [8]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape)

(30000, 45)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: flag a page for review_for_refresh when it is BOTH stale (no update in 180+ days) AND still visible (500+ impressions in the last 90 days). The score is the page's raw impressions_90d, so among qualifying pages, the ones with the most traffic to protect rank first. Everything else gets monitor.

Reason codes: stale_visible_page (matched both conditions) and no_action_needed (did not match).

Signal 1 — Staleness (behind FlyRank's refresh flags): checked whether decline rate rises as pages get staler.

Signal 2 — CTR vs position (behind FlyRank's CTR-fix logic): checked whether CTR falls as position tier gets worse.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#Signal 1
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, 180, 365, 10000],
    labels=["<30d", "30-90d", "90-180d", "180-365d", "365d+"]
)
staleness_table = df.groupby("staleness_bucket").agg(
    n=("is_declining_label", "size"),
    decline_rate=("is_declining_label", "mean")
).round(3)
print(staleness_table)

                      n  decline_rate
staleness_bucket                     
<30d              20480         0.511
30-90d              175         0.589
90-180d            9171         0.611
180-365d            169         0.467
365d+                 5         0.600


/tmp/ipykernel_716/3494955746.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_table = df.groupby("staleness_bucket").agg(


In [10]:
#Signal 2
visible = df[df["impressions_90d"] >= 100]
ctr_position_table = visible.groupby("position_tier").agg(
    n=("ctr", "size"),
    mean_ctr=("ctr", "mean")
).sort_values("mean_ctr", ascending=False).round(4)
print(ctr_position_table)

                  n  mean_ctr
position_tier                
page_1         8633    0.3548
top_3           533    0.3341
striking       5903    0.2558
page_3_5       6058    0.1424
deep            879    0.0554


Signal 1 — Staleness → verdict: MIXED. n=20,480 for <30d (decline rate 0.511), n=175 for 30-90d (0.589), n=9,171 for 90-180d (0.611), n=169 for 180-365d (0.467), n=5 for 365d+ (0.600). Decline rate does rise from <30d to 90-180d, but then drops at 180-365d before rising again at 365d+ — and the 365d+ bucket has only 5 rows, too few to trust. There is no clean monotonic relationship between staleness and decline once pages pass 180 days. This is a useful negative: staleness alone is a weak signal past the 180-day mark, so my rule should not assume "staler = always worse" — it should rely more on combining staleness with visibility, which is what my rule already does.

Signal 2 — CTR vs position → verdict: CONFIRMED. n=8,633 for page_1 (mean CTR 0.3548), n=533 for top_3 (0.3341), n=5,903 for striking (0.2558), n=6,058 for page_3_5 (0.1424), n=879 for deep (0.0554). CTR drops steadily as position tier worsens, confirming that CTR should only be compared within the same tier, not across the whole dataset.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_stale"] = df["days_since_last_update"] >= 180
df["is_visible"] = df["impressions_90d"] >= 500
df["baseline_score"] = df["is_stale"].astype(int) * df["is_visible"].astype(int) * df["impressions_90d"]
df["reason_code"] = "stale_visible_page"
df["action"] = "review_for_refresh"
df.loc[~(df["is_stale"] & df["is_visible"]), ["reason_code", "action"]] = ["no_action_needed", "monitor"]

queue = df.sort_values("baseline_score", ascending=False)[
    ["content_id", "client_id", "baseline_score", "reason_code", "action",
     "impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]
]
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")
queue.head(20)

Wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,review_for_refresh,61678,194,19.7,0.15,down
16514,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,review_for_refresh,59472,194,24.8,0.13,down
7021,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,review_for_refresh,25715,194,22.2,0.23,down
21268,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,review_for_refresh,13299,193,10.5,0.49,down
11489,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,review_for_refresh,7812,194,39.0,0.01,down
12045,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,review_for_refresh,7558,193,17.9,0.20,down
698,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,review_for_refresh,4590,194,31.0,0.00,down
5327,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,review_for_refresh,4556,194,16.4,0.33,down
26810,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,review_for_refresh,4429,194,25.3,0.38,down
20837,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,review_for_refresh,1697,193,15.8,0.12,down


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)
top20

,content_id,client_id,baseline_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,review_for_refresh,61678,194,19.7,0.15,down
16514,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,review_for_refresh,59472,194,24.8,0.13,down
7021,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,review_for_refresh,25715,194,22.2,0.23,down
21268,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,review_for_refresh,13299,193,10.5,0.49,down
11489,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,review_for_refresh,7812,194,39.0,0.01,down
12045,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,review_for_refresh,7558,193,17.9,0.20,down
698,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,review_for_refresh,4590,194,31.0,0.00,down
5327,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,review_for_refresh,4556,194,16.4,0.33,down
26810,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,review_for_refresh,4429,194,25.3,0.38,down
20837,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,review_for_refresh,1697,193,15.8,0.12,down


Only 17 rows in the entire dataset match my rule's conditions (stale AND visible) and get review_for_refresh — the rule is narrow by design. Rows 18-20 in a top-20 sort fall through to no_action_needed/monitor since there aren't 20 qualifying pages. All 17 flagged rows:

1. content_cf56e2e2e282 — Action: review_for_refresh. Confidence: high — 61,678 impressions, 194 days stale. What would make it wrong: if this is an intentionally evergreen page.

2. content_7368877ea310 — Action: review_for_refresh. Confidence: high — 59,472 impressions, 194 days stale. What would make it wrong: same as above.

3. content_1bfaa38ff26c — Action: review_for_refresh. Confidence: high — 25,715 impressions, 194 days stale, CTR 0.23. What would make it wrong: if CTR is already acceptable for its position tier.

4. content_0a91db491d14 — Action: review_for_refresh. Confidence: medium-high — 13,299 impressions, avg_position 10.5 (already strong), CTR 0.49. What would make it wrong: this page already ranks well and has decent CTR — a refresh may not move the needle much; staleness alone triggered the flag.

5. content_5feee3994adb — Action: review_for_refresh. Confidence: medium — 7,812 impressions, CTR only 0.01 despite avg_position 39.0. What would make it wrong: unlikely to be wrong — low CTR and weak position together make this a genuine candidate.

6. content_c2d929d83eaa — Action: review_for_refresh. Confidence: medium — 7,558 impressions, CTR 0.20. What would make it wrong: if the low CTR is typical for its position tier rather than a real problem.

7. content_b16bd7307b39 — Action: review_for_refresh. Confidence: medium — 4,590 impressions, CTR 0.00, position 31. What would make it wrong: unlikely wrong — zero CTR at weak position is a strong signal.

8. content_fe16a55cd13d — Action: review_for_refresh. Confidence: medium — 4,556 impressions, CTR 0.33, position 16.4. What would make it wrong: CTR and position both look reasonable already — staleness may be the only real issue here.

9. content_ecb6215e79fd — Action: review_for_refresh. Confidence: medium — 4,429 impressions, CTR 0.38. What would make it wrong: CTR already looks strong, so refresh priority may be lower than the score suggests.

10. content_928af3e22c80 — Action: review_for_refresh. Confidence: medium-low — 1,697 impressions, close to threshold. What would make it wrong: low volume makes this a marginal case.

11. content_e3ff1b093148 — Action: review_for_refresh. Confidence: medium-low — 1,408 impressions, different client (client_d029fa3a95), 183 days stale (just above cutoff). What would make it wrong: sits right at the staleness threshold, so a small measurement error could flip it.

12. content_bdbec75c1148 — Action: review_for_refresh. Confidence: low — 1,316 impressions, trend_direction is "stable" not "down". What would make it wrong: the page isn't actually declining despite being stale and visible — staleness may not matter here.

13. content_7f116ae1f6f5 — Action: review_for_refresh. Confidence: low — 954 impressions, different client (client_9400f1b21c), 301 days stale. What would make it wrong: low volume, marginal priority.

14. content_77d4d5930e5e — Action: review_for_refresh. Confidence: low — 828 impressions. What would make it wrong: low volume, close to the visibility cutoff.

15. content_72496874f806 — Action: review_for_refresh. Confidence: low — 821 impressions, different client (client_4ec9599fc2), avg_position 5.8 (already excellent). What would make it wrong: ranks very well already — a refresh is unlikely to be the priority here.

16. content_6226ee6adc91 — Action: review_for_refresh. Confidence: low — 545 impressions, just above the 500 threshold. What would make it wrong: right at the cutoff, low confidence this is meaningfully different from an unflagged page at 490 impressions.

17. content_074ba6ead17b — Action: review_for_refresh. Confidence: low — 533 impressions, CTR 0.00, avg_position 48.0 (very weak). What would make it wrong: unlikely wrong on relevance, but very low volume means the actual business impact of fixing it is small.

Pattern: 13 of the 17 flagged rows belong to the same client (client_7f2253d7e2). This may mean that client genuinely has the most stale, high-traffic content in this sample, or it may mean my raw-impressions scoring formula favors whichever client happens to have the largest overall traffic, which could drown out smaller clients with equally urgent but lower-volume pages. This is a real limitation of the current score design, not a data error.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Weakest picks: rows 16 and 17 (content_6226ee6adc91, content_074ba6ead17b) sit closest to the 500-impression cutoff (545 and 533), making them the least confident flags — a page at 490 impressions would be treated completely differently despite being nearly identical. Row 12 (content_bdbec75c1148) is also weak because its trend_direction is "stable," not "down," meaning it may not actually be declining despite matching the stale+visible rule.

Leakage check: this rule uses only days_since_last_update and impressions_90d, both observed same-window signals knowable at decision time. No FlyRank product decision fields (health_score, priority_score, action_type, refresh flags) were used — they aren't shipped in this dataset. No future-window or label-derived inputs were used in the scoring formula; trend_direction appears only in the output table for review context, not as a scoring input.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.